# Deep Learning PR 2
## Adult Income Dataset - ANN Binary Classification

This project uses the Adult Income (Census Income) dataset from the UCI Machine Learning Repository. It has 48,842 records with details about people such as age, education, occupation and hours worked per week.

The target is the `income` column. We want to predict whether a person earns more than 50K per year (class 1) or 50K or less (class 0). This is a binary classification problem.

We will build an ANN (MLP) using TensorFlow/Keras and study these concepts step by step:
- Activation functions
- Weight initialization
- Loss functions (BCE, MSE, Weighted BCE, Focal Loss)
- Batch Normalization
- Optimizers

Every experiment changes only one thing at a time so the comparison stays fair.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import Markdown, display

os.makedirs('plots', exist_ok=True)
sns.set_style('whitegrid')
print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


# Task 1: Data Loading, Cleaning & Exploratory Data Analysis

**Objective:** Load the Adult Income dataset, clean it, explore it with plots and prepare it for the neural network.

**Expected output:** A clean, encoded and scaled train/test split, plus four EDA plots.

## 1.1 Load and inspect the dataset

In [2]:
df = pd.read_csv('adult.csv')

# The Kaggle file uses hyphens in column names (for example hours-per-week).
# I rename them so the names match the ones used in the project definition.
df.columns = df.columns.str.replace('-', '.')
df.rename(columns={'educational.num': 'education.num', 'gender': 'sex'}, inplace=True)

print('Shape:', df.shape)
print()
print(df.dtypes)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'adult.csv'

In [ ]:
# strip extra whitespace from all text columns
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].str.strip()

df.head()

Text columns in census data can have extra spaces, for example `' Private'` instead of `'Private'`. Python treats these as two different values, so the same category would be split into two. Stripping the spaces makes sure each category is written in only one way. It also makes sure that comparisons like `df['income'] == '>50K'` work correctly later.

## 1.2 Handle missing values

In [ ]:
df.replace('?', np.nan, inplace=True)
print(df.isnull().sum())

In [ ]:
print('Shape before dropping:', df.shape)
df.dropna(inplace=True)
print('Shape after dropping :', df.shape)

Only three columns have missing values: `workclass`, `occupation` and `native.country`. In the raw file they were written as `'?'`, so I first converted them to `NaN` and then counted them.

I decided to drop these rows instead of filling them with the mode because:
- All three columns are categorical and have many categories (for example occupation has 14).
- Mode imputation would put every missing value into the most common category (for example `Private` or `United-States`). That would make the most common category even bigger than it really is and could add wrong information.
- The number of rows lost is small compared to the full dataset, so we still have plenty of data to train on.

## 1.3 Drop redundant columns

In [ ]:
df.drop(columns=['fnlwgt', 'education'], inplace=True)
print('Shape:', df.shape)
print(df.columns.tolist())

- **fnlwgt** is the census sampling weight. It tells how many people in the population a row represents. It is a survey statistic and not a personal feature of the individual, so it does not help to predict income.
- **education** is the text version of a person's education level (Bachelors, HS-grad, and so on).
- **education.num** is the same information in numeric form (1 to 16), where a higher number means more education. Keeping both would duplicate the same information, so I keep `education.num` because it is already a number and keeps the order of education levels. It also avoids adding about 16 extra one-hot columns.

## 1.4 Encode the target

In [ ]:
df['income'] = (df['income'] == '>50K').astype(int)
print(df['income'].value_counts())
print()
print(df['income'].value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='income', data=df, palette=['#2C2C2C', '#CC0000'], hue='income', legend=False)
total = len(df)
for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(f'{int(height)} ({height / total * 100:.1f}%)',
                (bar.get_x() + bar.get_width() / 2, height),
                ha='center', va='bottom', fontsize=10)
ax.set_xticks([0, 1])
ax.set_xticklabels(['0 (<=50K)', '1 (>50K)'])
plt.title('Class Distribution of Income')
plt.xlabel('Income class')
plt.ylabel('Number of people')
plt.savefig('plots/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

The dataset is imbalanced. Roughly 75% of the people belong to class 0 (income <=50K) and roughly 25% belong to class 1 (income >50K), which is about 3 to 1. The exact counts are shown on the bars above.

Because of this, accuracy alone can be misleading. A model that always predicts class 0 would already get about 75% accuracy without learning anything. So I will also track **Precision, Recall and F1-score for class 1** in every experiment.
- Precision: out of all people predicted as >50K, how many really earn >50K.
- Recall: out of all people who really earn >50K, how many the model found.
- F1-score: a single number that balances precision and recall.

## 1.5 Exploratory visualisations

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x='age', hue='income', bins=30, palette=['#2C2C2C', '#CC0000'])
plt.title('Age Distribution by Income')
plt.xlabel('Age (years)')
plt.ylabel('Number of people')
plt.savefig('plots/eda_age_by_income.png', dpi=150, bbox_inches='tight')
plt.show()

**Age plot:** The people with income <=50K (class 0) are spread over all ages with a large group in the twenties and early thirties. The people with income >50K (class 1) are mostly in the middle age range and there are very few of them at young ages. This suggests that age is a useful predictor, because people usually earn more after they gain work experience.

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x='income', y='hours.per.week', data=df, palette=['#2C2C2C', '#CC0000'], hue='income', legend=False)
plt.xticks([0, 1], ['0 (<=50K)', '1 (>50K)'])
plt.title('Hours per Week by Income')
plt.xlabel('Income class')
plt.ylabel('Hours per week')
plt.savefig('plots/eda_hours_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

**Hours per week plot:** The box for class 1 (>50K) sits higher than the box for class 0, which means people who earn more tend to work more hours per week. Both groups have many outliers, so hours worked alone cannot separate the two classes. It is a helpful feature but not enough by itself.

In [ ]:
plt.figure(figsize=(11, 5))
sns.countplot(x='education.num', hue='income', data=df, palette=['#2C2C2C', '#CC0000'])
plt.title('Education Level (education.num) by Income')
plt.xlabel('education.num (1 = lowest, 16 = highest)')
plt.ylabel('Number of people')
plt.legend(title='income', labels=['<=50K', '>50K'])
plt.savefig('plots/eda_education_countplot.png', dpi=150, bbox_inches='tight')
plt.show()

**Education plot:** At the lower education levels almost everyone is in class 0. As `education.num` increases, the share of class 1 (>50K) becomes larger and at the highest levels it becomes a big part of each bar. This shows that education is a strong predictor of income.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap of Numeric Columns')
plt.savefig('plots/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**Correlation heatmap:** Look at the `income` row (or column). The numeric features with the highest positive correlation with income are the ones with the darkest red shade, which are education.num, age, hours.per.week and capital.gain. All correlations are moderate or low, so no single numeric feature explains income on its own. The categorical features also carry a lot of information, so a model that can combine many features (like an ANN) makes sense.

## 1.6 Encode categorical variables and scale numeric variables

In [ ]:
# native.country has many rare countries, so group the rare ones into 'Other'
country_counts = df['native.country'].value_counts()
rare_countries = country_counts[country_counts < 100].index
df['native.country'] = df['native.country'].replace(rare_countries, 'Other')
print(df['native.country'].value_counts())

In [ ]:
X = df.drop('income', axis=1)
y = df['income']

X = pd.get_dummies(X, drop_first=True, dtype=int)
print('Columns after one-hot encoding:', X.shape[1])
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# scale only the numeric columns, not the one-hot columns
num_cols = ['age', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# convert to numpy arrays for Keras
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

In [ ]:
print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape :', y_test.shape)
print()
print('Full data class ratio :', np.round(y.value_counts(normalize=True).values, 4))
print('Train class ratio     :', np.round(np.bincount(y_train) / len(y_train), 4))
print('Test class ratio      :', np.round(np.bincount(y_test) / len(y_test), 4))

- **One-hot encoding:** A neural network can only work with numbers. Columns like `occupation` or `marital.status` contain text categories. One-hot encoding turns each category into its own 0/1 column. We do not use label encoding (1, 2, 3...) because that would tell the network that one category is "bigger" than another, which is not true. `drop_first=True` removes one column per feature, since it can be worked out from the others.
- **Scaling numeric columns:** Age is between 17 and 90 but capital.gain goes up to 99,999. Features with large values would dominate the weight updates and make training slow and unstable. StandardScaler brings each numeric column to mean 0 and standard deviation 1. I fit the scaler on the training data only and then applied it to the test data, so no information from the test set leaks into training.
- **Binary OHE columns are not scaled:** They are already only 0 or 1, which is a small range. Scaling them would only make them harder to interpret and does not help.
- **stratify=y:** It keeps the same class ratio (about 75/25) in both the train and test set. Without it, the minority class could be under-represented in one of the sets by chance, and the test metrics would be less reliable. The class ratios printed above confirm this.

# Task 2: MLP Architecture - Baseline ANN

**Objective:** Create one reusable `build_ann()` function and train a baseline ANN with it.

**Expected output:** Model summary, training curves, classification report and confusion matrix for the baseline.

## 2.1 The reusable build_ann() function

All later experiments call this function and change only one parameter at a time. Besides accuracy, I also add Precision and Recall as Keras metrics. This lets me calculate the validation F1-score of class 1 for every epoch from the training history later.

In [ ]:
def build_ann(input_dim, hidden_units=[128, 64], activation='relu',
              initializer='glorot_uniform', use_batch_norm=False,
              optimizer='adam', loss='binary_crossentropy'):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(input_dim,)))

    for units in hidden_units:
        model.add(keras.layers.Dense(units, kernel_initializer=initializer))
        if use_batch_norm:
            model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Activation(activation))

    model.add(keras.layers.Dense(1, activation='sigmoid'))

    model.compile(optimizer=optimizer, loss=loss,
                  metrics=['accuracy',
                           keras.metrics.Precision(name='precision'),
                           keras.metrics.Recall(name='recall')])
    return model

## 2.2 Baseline ANN architecture

In [ ]:
keras.utils.set_random_seed(42)
model_base = build_ann(input_dim=X_train.shape[1])
model_base.summary()

In [ ]:
# manual check of the parameter count
input_dim = X_train.shape[1]
layer1 = input_dim * 128 + 128
layer2 = 128 * 64 + 64
layer3 = 64 * 1 + 1
print('Layer 1 params:', layer1)
print('Layer 2 params:', layer2)
print('Output params :', layer3)
print('Total (manual):', layer1 + layer2 + layer3)
print('Total (Keras) :', model_base.count_params())

**What is an MLP?** An MLP (Multi-Layer Perceptron) is a fully connected neural network. Every neuron in one layer is connected to every neuron in the next layer. Each connection has a weight, and each neuron adds a bias and passes the result through an activation function. Our baseline is Input -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(1, sigmoid). The summary shows the Activation layers separately with 0 parameters, because only the Dense layers have weights.

**Why this dataset suits an ANN:** The data is structured tabular data with numeric features and categorical features that were converted to numbers. Each input neuron receives one scaled feature and the hidden layers can combine them.

**Why sigmoid and not softmax?** The output has to be the probability that the person earns more than 50K. Sigmoid takes one number and squashes it into the range (0, 1), so it can be read directly as P(class 1). Softmax is meant for several classes and needs one neuron per class. For two classes, softmax with two neurons gives the same result as sigmoid with one neuron but needs more parameters.

**Why one output neuron is enough:** The probability of class 0 is just 1 minus the probability of class 1, so a single neuron carries all the information.

## 2.3 Train the baseline

In [ ]:
history_base = model_base.fit(X_train, y_train, epochs=50, batch_size=64,
                              validation_split=0.1, verbose=1)

In [ ]:
epochs = range(1, 51)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history_base.history['loss'], color='#CC0000', label='Training loss')
axes[0].plot(epochs, history_base.history['val_loss'], color='#2C2C2C', label='Validation loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary cross-entropy loss')
axes[0].legend()

axes[1].plot(epochs, history_base.history['accuracy'], color='#CC0000', label='Training accuracy')
axes[1].plot(epochs, history_base.history['val_accuracy'], color='#2C2C2C', label='Validation accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

fig.suptitle('Baseline ANN — Training Curves (ReLU · Glorot Uniform · Adam · BCE)', fontsize=13)
plt.savefig('plots/baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Evaluate the baseline with full metrics

In [ ]:
y_pred_prob = model_base.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))

acc_base = accuracy_score(y_test, y_pred)
prec_base = precision_score(y_test, y_pred)
rec_base = recall_score(y_test, y_pred)
f1_base = f1_score(y_test, y_pred)
print('Baseline test accuracy        :', round(acc_base, 4))
print('Baseline precision (class 1)  :', round(prec_base, 4))
print('Baseline recall (class 1)     :', round(rec_base, 4))
print('Baseline F1-score (class 1)   :', round(f1_base, 4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['0 (<=50K)', '1 (>50K)'],
            yticklabels=['0 (<=50K)', '1 (>50K)'])
plt.title('Baseline ANN - Confusion Matrix (Test Set)')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.savefig('plots/baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

The overall accuracy looks good, but it mostly comes from class 0, which is the majority class. The numbers that really tell us how well the model handles the people who earn more than 50K are the **Precision, Recall and F1-score of class 1**, printed above and visible in the second row of the confusion matrix. These will be compared with all later models. All the metrics of the baseline are also collected in the final results table in Task 7.

## 2.5 Why tabular data suits an ANN

Structured tabular data is made of columns of numeric features and categorical features. After one-hot encoding and scaling, every column becomes a number, so each input neuron of the MLP can receive exactly one feature. This is different from images (where a CNN is better because it looks at neighbouring pixels) or sequences (where an RNN is better because the order matters).

The hidden layers learn weighted combinations of the features and, because of the activation functions, they can also learn non-linear relationships and interactions, for example the combined effect of age and education, or hours worked and occupation. A simple logistic regression can only use each feature in a straight-line way, so it cannot capture these interactions easily.

# Task 3: Activation Functions - Comparison Experiment

**Objective:** Compare ReLU, Tanh, Sigmoid and ELU as hidden-layer activations while everything else stays the same.

**Expected output:** A 2x2 comparison plot, a dead neuron check for ReLU, a gradient flow check for sigmoid and a summary table.

## 3.1 Train 4 models with different hidden activations

Only the `activation` parameter changes. The random seed is set to 42 before each model so all four start from the same random state.

In [ ]:
keras.utils.set_random_seed(42)
model_relu = build_ann(input_dim=X_train.shape[1], activation='relu')
history_relu = model_relu.fit(X_train, y_train, epochs=50, batch_size=64,
                              validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_tanh = build_ann(input_dim=X_train.shape[1], activation='tanh')
history_tanh = model_tanh.fit(X_train, y_train, epochs=50, batch_size=64,
                              validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_sigmoid = build_ann(input_dim=X_train.shape[1], activation='sigmoid')
history_sigmoid = model_sigmoid.fit(X_train, y_train, epochs=50, batch_size=64,
                                    validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_elu = build_ann(input_dim=X_train.shape[1], activation='elu')
history_elu = model_elu.fit(X_train, y_train, epochs=50, batch_size=64,
                            validation_split=0.1, verbose=0)

print('All 4 activation models trained.')

## 3.2 Activation comparison - 4-panel plot

The F1-score in each title is the validation F1-score of class 1 at the last epoch. It is calculated from the validation precision and recall stored in the history.

In [ ]:
histories_act = {'ReLU': history_relu, 'Tanh': history_tanh,
                 'Sigmoid': history_sigmoid, 'ELU': history_elu}
models_act = {'ReLU': model_relu, 'Tanh': model_tanh,
              'Sigmoid': model_sigmoid, 'ELU': model_elu}

val_acc_act = {}
val_f1_act = {}
epochs = range(1, 51)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

i = 0
for name in histories_act:
    h = histories_act[name].history
    p = np.array(h['val_precision'])
    r = np.array(h['val_recall'])
    f1_curve = 2 * p * r / (p + r + 1e-8)

    val_acc_act[name] = h['val_accuracy'][-1]
    val_f1_act[name] = f1_curve[-1]

    axes[i].plot(epochs, h['accuracy'], color='#CC0000', label='Training accuracy')
    axes[i].plot(epochs, h['val_accuracy'], color='#2C2C2C', label='Validation accuracy')
    axes[i].axhline(0.75, color='gray', linestyle='--', label='75% majority-class baseline')
    axes[i].set_title(f'{name} - Val Acc: {val_acc_act[name]:.4f} | Val F1 (class 1): {val_f1_act[name]:.4f}')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Accuracy')
    axes[i].legend(loc='lower right')
    i = i + 1

fig.suptitle('Activation Function Comparison (Adult Income ANN)', fontsize=14)
plt.savefig('plots/activation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Dead neuron check - ReLU

A dead ReLU neuron is a neuron that outputs 0 for every input. Here I take the output of the first hidden layer (after the ReLU) for 500 test samples and check how many outputs are exactly 0.

In [ ]:
intermediate = keras.Model(inputs=model_relu.inputs, outputs=model_relu.layers[1].output)
activations = intermediate.predict(X_test[:500])

dead_fraction = np.mean(activations == 0)
mean_per_neuron = activations.mean(axis=0)
fully_dead_neurons = np.sum(activations.max(axis=0) == 0)

print('Activation matrix shape:', activations.shape)
print('dead_fraction (share of all outputs that are exactly 0):', round(dead_fraction, 4))
print('Neurons that were 0 for all 500 samples:', fully_dead_neurons, 'out of', activations.shape[1])

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(mean_per_neuron, bins=30, color='#CC0000', edgecolor='black')
plt.title('ReLU First-Layer Activation Distribution')
plt.xlabel('Mean activation of the neuron (500 test samples)')
plt.ylabel('Number of neurons')
plt.savefig('plots/relu_activation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

The value of `dead_fraction` for this run is printed in the cell above, together with the number of neurons that were zero for all 500 samples.

**What is a dead ReLU neuron?** ReLU is defined as max(0, x). If the input of a neuron is negative, the output is 0 and the gradient is also 0. If a neuron's input becomes negative for all samples (for example after a large weight update), its output is always 0, it gets no gradient and it cannot recover. That neuron is "dead" and no longer helps the network.

**Note on the fraction:** Not every zero means a dead neuron. Even a healthy ReLU neuron outputs 0 for the samples where its input is negative, so `dead_fraction` includes these normal zeros. The number of neurons that were 0 for all samples is a better sign of truly dead neurons.

**How ELU avoids this:** ELU gives output x for x > 0 and alpha * (e^x - 1) for x < 0. For negative inputs it still gives a small negative output and a non-zero gradient, so the neuron can still learn and come back to life. The cost is that it needs to calculate an exponential, which is slightly slower.

## 3.4 Gradient flow check - Sigmoid hidden layers

I take one batch of 64 training samples, calculate the loss with `tf.GradientTape` and look at the mean absolute gradient of the weights in each Dense layer of the sigmoid model.

In [ ]:
x_batch = tf.constant(X_train[:64], dtype=tf.float32)
y_batch = tf.constant(y_train[:64].reshape(-1, 1), dtype=tf.float32)

with tf.GradientTape() as tape:
    preds = model_sigmoid(x_batch, training=True)
    loss_value = tf.reduce_mean(keras.losses.binary_crossentropy(y_batch, preds))

grads = tape.gradient(loss_value, model_sigmoid.trainable_variables)

# trainable_variables order: kernel1, bias1, kernel2, bias2, kernel3, bias3
grad_layer1 = np.mean(np.abs(grads[0].numpy()))
grad_layer2 = np.mean(np.abs(grads[2].numpy()))
grad_output = np.mean(np.abs(grads[4].numpy()))

print('Mean |gradient| Layer 1 :', grad_layer1)
print('Mean |gradient| Layer 2 :', grad_layer2)
print('Mean |gradient| Output  :', grad_output)
print('Output / Layer 1 ratio  :', grad_output / grad_layer1)

In [ ]:
plt.figure(figsize=(8, 4))
layer_names = ['Layer 1', 'Layer 2', 'Output']
grad_values = [grad_layer1, grad_layer2, grad_output]
plt.barh(layer_names, grad_values, color=['#CC0000', '#CC0000', '#2C2C2C'])
plt.gca().invert_yaxis()
plt.title('Gradient Magnitude by Layer — Sigmoid Hidden Activations')
plt.xlabel('Mean |gradient| of the weights')
plt.ylabel('Dense layer')
plt.savefig('plots/sigmoid_gradient_flow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if grad_output > 10 * grad_layer1:
    print('In this run the Layer 1 gradient is much smaller than the output layer gradient (vanishing gradient effect is visible).')
else:
    print('In this run the Layer 1 gradient is not orders of magnitude smaller than the output layer gradient.')
    print('With only two hidden layers the shrinking effect is small, and this model was already trained for 50 epochs.')

**What are vanishing gradients?** During backpropagation the gradient of the loss is passed backwards from the output layer to the first layer by multiplying by the derivative of each layer. If these derivatives are small, the gradient becomes smaller and smaller as it moves back, so the early layers receive a tiny gradient and learn very slowly.

**Why sigmoid causes it:** The derivative of sigmoid is sigma'(x) = sigma(x) * (1 - sigma(x)). The largest value this can have is 0.25 (at x = 0), and it gets close to 0 when the input is very large or very small. When we multiply several numbers that are at most 0.25, the result shrinks exponentially with the number of layers. For example, two sigmoid layers can already shrink the gradient by a factor of up to 0.25 x 0.25 = 0.0625.

The printed message above tells what happened in this run. If the Layer 1 gradient is orders of magnitude smaller than the output gradient, vanishing gradients are clearly happening. Our network is shallow (only 2 hidden layers), so the effect is much weaker than it would be in a deep network.

## 3.5 Activation summary table

In [ ]:
info = [
    ['ReLU', 'max(0, x)', '[0, infinity)', 'No', 'Low', 'High'],
    ['Tanh', '(e^x - e^-x) / (e^x + e^-x)', '(-1, 1)', 'Yes', 'Medium', 'Low'],
    ['Sigmoid (hidden)', '1 / (1 + e^-x)', '(0, 1)', 'No', 'High', 'Low'],
    ['ELU', 'x if x > 0, else alpha * (e^x - 1)', '(-alpha, infinity)', 'Roughly', 'Low', 'Very low'],
]
names_in_results = ['ReLU', 'Tanh', 'Sigmoid', 'ELU']

table = '| Activation | Formula | Output Range | Zero-Centred | Vanishing Gradient Risk | Dead Neuron Risk | Val Accuracy (this run) | Val F1 class 1 (this run) |\n'
table += '|---|---|---|---|---|---|---|---|\n'
for i in range(4):
    row = info[i]
    key = names_in_results[i]
    table += f'| {row[0]} | {row[1]} | {row[2]} | {row[3]} | {row[4]} | {row[5]} | {val_acc_act[key]:.4f} | {val_f1_act[key]:.4f} |\n'
table += '| Sigmoid (output layer only) | 1 / (1 + e^-x) | (0, 1) | No | Not a problem for one output neuron | Not applicable | used in all models | used in all models |\n'

display(Markdown(table))

best_activation = max(val_f1_act, key=val_f1_act.get)
print('Best activation in this run (highest validation F1 for class 1):', best_activation)
print('Its validation accuracy:', round(val_acc_act[best_activation], 4))

**Recommendation:** The recommended activation for this dataset is the one with the highest validation F1-score for class 1 in the table above (printed as `best_activation`), because F1 of the minority class matters more than accuracy here. I use this together with the theory:
- ReLU and ELU do not saturate for positive inputs, so their gradients stay healthy. They are also cheap to compute.
- Sigmoid in hidden layers is the riskiest because its derivative is at most 0.25.
- Tanh is zero-centred and works well for shallow networks like ours, so it can be close to ReLU/ELU.

In a shallow 2-hidden-layer network the differences between activations can be small, and one run with one random seed is not enough to say that the small differences are definitely real. ReLU is the standard default for tabular ANNs.

The sigmoid in the **output layer** is always correct for binary classification, whatever activation is used in the hidden layers, because it converts the final value into a probability.

# Task 4: Weight Initialization Techniques

**Objective:** Compare five weight initializers with a ReLU network and see why initialization matters.

**Expected output:** A convergence comparison, a failure plot for zero initialization and weight histograms before and after training.

## 4.2 Why initialization matters

**Zero initialization and the symmetry problem:** If all weights start at zero, every neuron in a layer computes exactly the same output, gets exactly the same gradient and is updated in exactly the same way. The neurons never become different from each other, so the whole layer behaves like one neuron, however wide it is. With ReLU and zero weights it is even worse, because the hidden outputs are all 0 and the hidden weights receive no gradient at all.

**Xavier / Glorot initialization:** The weights are drawn from a distribution with variance Var = 2 / (fan_in + fan_out), where fan_in is the number of inputs of the layer and fan_out is the number of outputs. It keeps the variance of the activations and gradients about the same across layers. It is designed for tanh and sigmoid.

**He initialization:** The weights use variance Var = 2 / fan_in. ReLU sets about half of the activations to zero, which cuts the variance of the signal in half. The extra factor of 2 makes up for this so the signal does not shrink layer after layer. This is why He initialization is the natural choice for ReLU networks.

Keras has a uniform and a normal version of each: `glorot_uniform`, `glorot_normal`, `he_uniform`, `he_normal`. They have the same target variance and differ only in the shape of the distribution.

In the next cell I build all five models and save the first-layer weights of `he_normal` and `glorot_uniform` **before** training, because these are needed for the histogram in 4.5.

## 4.1 Train 5 models with different initializers

In [ ]:
# glorot_uniform
keras.utils.set_random_seed(42)
model_glorot_uniform = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='glorot_uniform')
w_glorot_before = model_glorot_uniform.layers[0].get_weights()[0].flatten()
history_glorot_uniform = model_glorot_uniform.fit(X_train, y_train, epochs=50, batch_size=64,
                                                  validation_split=0.1, verbose=0)
w_glorot_after = model_glorot_uniform.layers[0].get_weights()[0].flatten()

# glorot_normal
keras.utils.set_random_seed(42)
model_glorot_normal = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='glorot_normal')
history_glorot_normal = model_glorot_normal.fit(X_train, y_train, epochs=50, batch_size=64,
                                                validation_split=0.1, verbose=0)

# he_uniform
keras.utils.set_random_seed(42)
model_he_uniform = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_uniform')
history_he_uniform = model_he_uniform.fit(X_train, y_train, epochs=50, batch_size=64,
                                          validation_split=0.1, verbose=0)

# he_normal
keras.utils.set_random_seed(42)
model_he_normal = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal')
w_he_before = model_he_normal.layers[0].get_weights()[0].flatten()
history_he_normal = model_he_normal.fit(X_train, y_train, epochs=50, batch_size=64,
                                        validation_split=0.1, verbose=0)
w_he_after = model_he_normal.layers[0].get_weights()[0].flatten()

# zeros
keras.utils.set_random_seed(42)
model_zeros = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='zeros')
history_zeros = model_zeros.fit(X_train, y_train, epochs=50, batch_size=64,
                                validation_split=0.1, verbose=0)

print('All 5 initializer models trained.')

## 4.3 Convergence speed comparison

In [ ]:
histories_init = {'glorot_uniform': history_glorot_uniform, 'glorot_normal': history_glorot_normal,
                  'he_uniform': history_he_uniform, 'he_normal': history_he_normal,
                  'zeros': history_zeros}
models_init = {'glorot_uniform': model_glorot_uniform, 'glorot_normal': model_glorot_normal,
               'he_uniform': model_he_uniform, 'he_normal': model_he_normal,
               'zeros': model_zeros}
colors = {'glorot_uniform': '#CC0000', 'glorot_normal': '#2C2C2C', 'he_uniform': '#1f77b4',
          'he_normal': '#2ca02c', 'zeros': '#ff7f0e'}
styles = {'glorot_uniform': '-', 'glorot_normal': '--', 'he_uniform': '-.',
          'he_normal': ':', 'zeros': '-'}

epochs = range(1, 51)
epoch_to_82 = {}
val_acc_init = {}
val_f1_init = {}

plt.figure(figsize=(12, 6))
for name in histories_init:
    h = histories_init[name].history
    val_acc = np.array(h['val_accuracy'])
    val_acc_init[name] = val_acc[-1]

    p = np.array(h['val_precision'])
    r = np.array(h['val_recall'])
    val_f1_init[name] = (2 * p * r / (p + r + 1e-8))[-1]

    label = name
    if name != 'zeros':
        crossed = np.where(val_acc >= 0.82)[0]
        if len(crossed) > 0:
            first_epoch = crossed[0] + 1
            epoch_to_82[name] = first_epoch
            plt.axvline(first_epoch, color=colors[name], linestyle=':', alpha=0.6)
            label = f'{name} (82% at epoch {first_epoch})'
        else:
            epoch_to_82[name] = None
            label = f'{name} (never reached 82%)'
    plt.plot(epochs, val_acc, color=colors[name], linestyle=styles[name], label=label)

plt.axhline(0.84, color='gray', linestyle='--', label='84% target benchmark')
plt.title('Weight Initialisation — Convergence Speed Comparison (Adult Income Dataset)')
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend(loc='lower right')
plt.savefig('plots/initialiser_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print('First epoch with validation accuracy >= 82%:')
print(epoch_to_82)
print()
print('Final validation accuracy of each initializer:')
for name in val_acc_init:
    print(name, round(val_acc_init[name], 4))

The vertical dotted lines mark the first epoch at which each non-zero initializer crossed 82% validation accuracy, and the same values are printed under the plot. If an initializer never crossed 82%, the legend says so. The zeros model is not marked because it is the failure case.

## 4.4 Zeros failure demonstration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history_zeros.history['loss'], color='#CC0000')
axes[0].set_title('Zeros initialisation - Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss')

axes[1].plot(epochs, history_zeros.history['val_accuracy'], color='#2C2C2C', label='Validation accuracy')
axes[1].axhline(0.75, color='gray', linestyle='--', label='75% majority-class baseline')
axes[1].set_title('Zeros initialisation - Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation accuracy')
axes[1].legend()

plt.savefig('plots/zeros_failure.png', dpi=150, bbox_inches='tight')
plt.show()

# check what the zeros model predicts
zeros_probs = model_zeros.predict(X_test[:1000]).flatten()
print('Number of different output values for 1000 samples:', len(np.unique(np.round(zeros_probs, 6))))
print('Min output:', zeros_probs.min(), ' Max output:', zeros_probs.max())
print('Final validation accuracy:', round(history_zeros.history['val_accuracy'][-1], 4))

The accuracy of the zeros model stays flat at about the majority-class level (the printed values above show the exact numbers). The model gives the same output for every sample, so the number of different output values printed above is 1 or close to it. It has learned nothing about the features and just predicts a constant value close to the majority class.

The reason is the symmetry problem. All hidden weights start at zero, so every hidden neuron gives the same output (0 after ReLU) and the hidden weights get the same (zero) gradient. Only the bias of the output neuron can learn, and it just learns the average class probability. The network never breaks symmetry. This shows that weight initialization is not a small detail. It decides whether the network can learn at all.

## 4.5 Weight distribution visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].hist(w_he_before, bins=50, color='#CC0000', edgecolor='black')
axes[0, 0].set_title('he_normal - Before Training')

axes[0, 1].hist(w_he_after, bins=50, color='#2C2C2C', edgecolor='black')
axes[0, 1].set_title('he_normal - After 50 Epochs')

axes[1, 0].hist(w_glorot_before, bins=50, color='#CC0000', edgecolor='black')
axes[1, 0].set_title('glorot_uniform - Before Training')

axes[1, 1].hist(w_glorot_after, bins=50, color='#2C2C2C', edgecolor='black')
axes[1, 1].set_title('glorot_uniform - After 50 Epochs')

for ax in axes.flatten():
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Count')

fig.suptitle('Weight Distributions — Initial vs Trained', fontsize=14)
plt.savefig('plots/weight_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('he_normal        before: std =', round(w_he_before.std(), 4), ' after: std =', round(w_he_after.std(), 4))
print('glorot_uniform   before: std =', round(w_glorot_before.std(), 4), ' after: std =', round(w_glorot_after.std(), 4))
print('Theory: he_normal std =', round(np.sqrt(2 / X_train.shape[1]), 4),
      ' glorot_uniform std =', round(np.sqrt(2 / (X_train.shape[1] + 128)), 4))

Before training, the histograms should match the theory. The he_normal weights form a bell shape with standard deviation sqrt(2 / fan_in), and the glorot_uniform weights are flat between two limits with a standard deviation sqrt(2 / (fan_in + fan_out)). Because 2 / fan_in is larger than 2 / (fan_in + fan_out), the He distribution is wider than the Glorot one. The printed standard deviations can be compared with the theory values in the last line of the cell above.

After 50 epochs, both histograms spread out and their shapes change from their original shape. This is because backpropagation keeps changing the weights depending on the data. The initializer only decides the starting point. It helps the network start with healthy signals and gradients, and after that the training reshapes the weights.

# Task 5: Loss Functions

**Objective:** Compare Binary Cross-Entropy, MSE, Weighted BCE and Focal Loss, with a focus on the minority class (>50K).

**Expected output:** F1 comparison plots and a summary table of the four losses.

## 5.1 Binary Cross-Entropy - the correct loss

In [ ]:
# the default loss in build_ann() is the last default value
print('Defaults of build_ann():', build_ann.__defaults__)
print('The baseline model_base was built with the default loss: binary_crossentropy')

The baseline uses `loss='binary_crossentropy'` (the last default of `build_ann()`).

**BCE formula:** L = -[ y * log(p) + (1 - y) * log(1 - p) ]

Here p is the sigmoid output (predicted probability of class 1) and y is the true label (0 or 1).
- If y = 1, only the first term stays, so the loss is -log(p). It is small when p is close to 1 and becomes very large when p is close to 0.
- If y = 0, only the second term stays, so the loss is -log(1 - p). It is small when p is close to 0 and becomes very large when p is close to 1.

So a confident wrong prediction (for example p close to 1 when y = 0) gets a very big penalty, because log(1 - p) goes towards minus infinity. BCE is the negative log-likelihood of a Bernoulli distribution (a coin flip with probability p), which is why it is the natural loss for binary classification.

## 5.2 MSE as classification loss - wrong tool experiment

In [ ]:
keras.utils.set_random_seed(42)
model_mse = build_ann(input_dim=X_train.shape[1], loss='mean_squared_error',
                      activation='relu', initializer='he_normal')
history_mse = model_mse.fit(X_train, y_train, epochs=50, batch_size=64,
                            validation_split=0.1, verbose=0)
print('MSE model trained.')

In [ ]:
epochs = range(1, 51)

p = np.array(history_mse.history['val_precision'])
r = np.array(history_mse.history['val_recall'])
f1_curve_mse = 2 * p * r / (p + r + 1e-8)

p = np.array(history_base.history['val_precision'])
r = np.array(history_base.history['val_recall'])
f1_curve_base = 2 * p * r / (p + r + 1e-8)

p = np.array(history_he_normal.history['val_precision'])
r = np.array(history_he_normal.history['val_recall'])
f1_curve_he = 2 * p * r / (p + r + 1e-8)

plt.figure(figsize=(11, 6))
plt.plot(epochs, f1_curve_base, color='#CC0000', label='model_base (BCE, glorot_uniform)')
plt.plot(epochs, f1_curve_mse, color='#2C2C2C', linestyle='--', label='model_mse (MSE, he_normal)')
plt.plot(epochs, f1_curve_he, color='#1f77b4', linestyle=':', label='BCE with he_normal (from Task 4)')
plt.title('Validation F1-score (class 1): BCE vs MSE')
plt.xlabel('Epoch')
plt.ylabel('Validation F1-score (class 1)')
plt.legend()
plt.savefig('plots/loss_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Final validation F1 (class 1) - model_base :', round(f1_curve_base[-1], 4))
print('Final validation F1 (class 1) - model_mse  :', round(f1_curve_mse[-1], 4))
print('Final validation F1 (class 1) - BCE he_normal:', round(f1_curve_he[-1], 4))

In [ ]:
y_prob_mse = model_mse.predict(X_test).flatten()
y_pred_mse = (y_prob_mse > 0.5).astype(int)
print(classification_report(y_test, y_pred_mse))
f1_mse = f1_score(y_test, y_pred_mse)

The task asks for a comparison against `model_base`, but `model_mse` also uses `he_normal` while `model_base` uses `glorot_uniform`. So I added a third line, the BCE model with `he_normal` from Task 4. The difference between the third line and the MSE line is only the loss function.

**Why BCE is more appropriate than MSE for this problem:**
- MSE treats the sigmoid output as a number in a regression problem. It does not punish a confident wrong prediction as strongly as BCE does. The MSE for a wrong prediction is at most 1, while the BCE goes to infinity.
- The gradient of MSE with respect to the sigmoid input contains the term sigma(x) * (1 - sigma(x)). When the output is saturated (close to 0 or 1) this term is almost 0, so the gradient vanishes even when the prediction is wrong and the model learns slowly.
- For BCE with sigmoid the gradient with respect to the logit is simply (p - y). It is large when the prediction is wrong and small when it is right, and it never vanishes for wrong predictions.
- On imbalanced data, MSE also tends to be dominated by the majority class, so the minority class F1 is usually affected the most.

The plot and the printed values show what happened in this run.

## 5.3 Weighted Binary Cross-Entropy - handling class imbalance

In [ ]:
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: weights[0], 1: weights[1]}
print('Class weights:', class_weight_dict)

keras.utils.set_random_seed(42)
model_wbce = build_ann(input_dim=X_train.shape[1], loss='binary_crossentropy',
                       activation='relu', initializer='he_normal')
history_wbce = model_wbce.fit(X_train, y_train, epochs=50, batch_size=64,
                              validation_split=0.1, class_weight=class_weight_dict, verbose=0)
print('Weighted BCE model trained.')

In [ ]:
y_prob_wbce = model_wbce.predict(X_test).flatten()
y_pred_wbce = (y_prob_wbce > 0.5).astype(int)
print(classification_report(y_test, y_pred_wbce))

prec_wbce = precision_score(y_test, y_pred_wbce)
rec_wbce = recall_score(y_test, y_pred_wbce)
f1_wbce = f1_score(y_test, y_pred_wbce)

metric_names = ['Precision', 'Recall', 'F1-score']
base_values = [prec_base, rec_base, f1_base]
wbce_values = [prec_wbce, rec_wbce, f1_wbce]

x_pos = np.arange(3)
plt.figure(figsize=(8, 5))
plt.bar(x_pos - 0.2, base_values, width=0.4, color='#2C2C2C', label='model_base (no weights)')
plt.bar(x_pos + 0.2, wbce_values, width=0.4, color='#CC0000', label='model_wbce (weighted BCE)')
for i in range(3):
    plt.text(x_pos[i] - 0.2, base_values[i] + 0.01, f'{base_values[i]:.3f}', ha='center')
    plt.text(x_pos[i] + 0.2, wbce_values[i] + 0.01, f'{wbce_values[i]:.3f}', ha='center')
plt.xticks(x_pos, metric_names)
plt.ylim(0, 1)
plt.title('Class 1 (>50K) Metrics on Test Set: Baseline vs Weighted BCE')
plt.ylabel('Score')
plt.legend()
plt.savefig('plots/weighted_bce_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**What class weights do:** With `class_weight='balanced'` the weight of each class is inversely proportional to how often it appears. The minority class (>50K) gets a weight larger than 1 and the majority class gets a weight smaller than 1 (the values are printed above). A mistake on a >50K person then counts more in the loss than a mistake on a <=50K person.

**Effect on the metrics:** The model is pushed to find more of the high earners, so **Recall of class 1 usually goes up**. As a side effect the model also labels some more people as >50K by mistake, so **Precision can go down**. F1 shows whether the trade is worth it overall. The bar chart shows the actual change in this run.

This trade-off is useful in income targeting, where missing a high earner can be more costly than a false positive.

## 5.4 Focal Loss - advanced imbalance handling

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_true = tf.reshape(y_true, tf.shape(y_pred))
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        weight = alpha * y_true * (1 - y_pred) ** gamma + (1 - alpha) * (1 - y_true) * y_pred ** gamma
        return tf.reduce_mean(weight * bce)
    return loss

In [ ]:
keras.utils.set_random_seed(42)
model_focal = build_ann(input_dim=X_train.shape[1], loss=focal_loss(gamma=2.0, alpha=0.25),
                        activation='relu', initializer='he_normal')
history_focal = model_focal.fit(X_train, y_train, epochs=50, batch_size=64,
                                validation_split=0.1, verbose=0)

y_prob_focal = model_focal.predict(X_test).flatten()
y_pred_focal = (y_prob_focal > 0.5).astype(int)
print(classification_report(y_test, y_pred_focal))

prec_focal = precision_score(y_test, y_pred_focal)
rec_focal = recall_score(y_test, y_pred_focal)
f1_focal = f1_score(y_test, y_pred_focal)

In [ ]:
recall_values = [rec_base, rec_wbce, rec_focal]
f1_values = [f1_base, f1_wbce, f1_focal]
model_labels = ['Baseline (BCE)', 'Weighted BCE', 'Focal Loss']

x_pos = np.arange(3)
plt.figure(figsize=(9, 5))
plt.bar(x_pos - 0.2, recall_values, width=0.4, color='#2C2C2C', label='Recall (class 1)')
plt.bar(x_pos + 0.2, f1_values, width=0.4, color='#CC0000', label='F1-score (class 1)')
for i in range(3):
    plt.text(x_pos[i] - 0.2, recall_values[i] + 0.01, f'{recall_values[i]:.3f}', ha='center')
    plt.text(x_pos[i] + 0.2, f1_values[i] + 0.01, f'{f1_values[i]:.3f}', ha='center')
plt.xticks(x_pos, model_labels)
plt.ylim(0, 1)
plt.title('Class 1 Recall and F1: Baseline vs Weighted BCE vs Focal Loss (Test Set)')
plt.ylabel('Score')
plt.legend()
plt.savefig('plots/focal_loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Focal Loss was introduced by Facebook AI (Lin et al., 2017) for object detection with extreme class imbalance. It multiplies the BCE by a factor (1 - p)^gamma, where p is the probability the model gives to the correct class:
- For an easy example (the model is already confident and right), (1 - p) is small, so the factor is very small and this example contributes almost nothing to the loss.
- For a hard, misclassified example, (1 - p) is large, so the factor is close to 1 and the example keeps its full loss.

So the training focuses on the hard examples instead of spending effort on the many easy majority-class samples. `gamma = 2` controls how strongly the easy examples are down-weighted. `alpha = 0.25` is the class balance weight, applied to the positive class in the code from the project definition. Note that this puts a **smaller** weight (0.25) on the >50K class and a bigger one (0.75) on the <=50K class, which is the opposite direction of class weighting. So focal loss with these default values is not guaranteed to improve minority-class recall. The bar chart above shows what happened in this run.

## 5.5 Loss function summary table

In [ ]:
losses_info = [
    ['BCE', '-[y log(p) + (1-y) log(1-p)]', 'Balanced binary classification', 'None', f1_base],
    ['Weighted BCE', '-[w1 y log(p) + w0 (1-y) log(1-p)]', 'Imbalanced binary classification', 'Class weights from class frequencies', f1_wbce],
    ['MSE (classification)', '(y - p)^2', 'Regression problems, not classification', 'None', f1_mse],
    ['Focal Loss', '-alpha (1-p)^gamma log(p) for class 1 and similar for class 0', 'Strong imbalance with many easy examples', 'Down-weights easy examples (gamma) and uses alpha', f1_focal],
]

table = '| Loss Function | Formula (simplified) | Best For | Imbalance Handling | Class-1 F1 on Adult (this run, test set) |\n'
table += '|---|---|---|---|---|\n'
for row in losses_info:
    table += f'| {row[0]} | {row[1]} | {row[2]} | {row[3]} | {row[4]:.4f} |\n'
display(Markdown(table))

f1_by_loss = {'BCE': f1_base, 'Weighted BCE': f1_wbce, 'MSE': f1_mse, 'Focal Loss': f1_focal}
best_loss = max(f1_by_loss, key=f1_by_loss.get)
print('Loss with the highest class-1 F1 in this run:', best_loss, round(f1_by_loss[best_loss], 4))

In [ ]:
loss_reasons = {
    'BCE': 'Plain BCE gave the best class-1 F1 here. It is simple and its gradient (p - y) is stable. Because the F1 is at the default 0.5 threshold, the other losses did not beat it in this run.',
    'Weighted BCE': 'Weighted BCE gave the best class-1 F1 here. It keeps the stable BCE gradient and adds a higher penalty for mistakes on the minority class, so the model finds more high earners without needing any special loss code.',
    'MSE': 'MSE gave the best class-1 F1 here, but this is unusual. It should be checked carefully because MSE is not designed for classification and BCE is normally the safer choice.',
    'Focal Loss': 'Focal Loss gave the best class-1 F1 here. Focusing on hard examples helped the minority class in this run, but it needs extra tuning of gamma and alpha.',
}
print('Recommended loss for the Adult Income ANN based on this run:', best_loss)
print(loss_reasons[best_loss])

**Recommendation:** The recommended loss is the one printed above, chosen by the highest class-1 F1 on the test set in this run. In general, MSE is the wrong tool for classification, and BCE-based losses are the right family. Between them, the choice depends on how much we value recall of the minority class versus overall balance. Since only one run and one seed were used, small differences between the losses should not be over-interpreted.

# Task 6: Batch Normalization

**Objective:** Add Batch Normalization to the ANN and study its effect on training, its position in the network and its learned parameters.

**Expected output:** Model summary, a training dynamics comparison, a position experiment and gamma/beta plots.

## 6.1 Build ANN with Batch Normalization

In [ ]:
keras.utils.set_random_seed(42)
model_bn = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dense(64, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model_bn.compile(optimizer='adam', loss='binary_crossentropy',
                 metrics=['accuracy',
                          keras.metrics.Precision(name='precision'),
                          keras.metrics.Recall(name='recall')])
model_bn.summary()

In the summary each BatchNormalization layer has 4 x (number of neurons) parameters: gamma, beta, moving_mean and moving_variance. Only gamma and beta are trainable. The moving mean and moving variance are non-trainable, because they are not learned by gradient descent but updated as running averages. That is why the summary shows a trainable and a non-trainable parameter count. For the 128-neuron layer this is 512 parameters, of which 256 are trainable and 256 are not.

## 6.2 Why Batch Normalization

While training, the weights of each layer keep changing, so the distribution of the inputs to the next layer keeps shifting. Every layer must keep adapting to a moving target. This is called **Internal Covariate Shift** (Ioffe and Szegedy, 2015).

**What Batch Normalization does:** It normalises the pre-activation values of each mini-batch to zero mean and unit variance, and then applies a learnable scale (gamma) and shift (beta):

BN(x) = gamma * (x - mu_B) / sqrt(sigma_B^2 + epsilon) + beta

- **gamma and beta** are learned during training, so the network can undo the normalisation if that is better for it.
- **moving mean and moving variance** are running averages of the batch means and variances collected during training.

**Training vs inference:** During training BN uses the mean and variance of the current mini-batch. During inference (prediction), there may be a single sample or a different batch, so BN uses the stored moving mean and moving variance instead. This makes the predictions the same no matter which other samples are in the batch.

**Why before the activation:** In the canonical order (Dense -> BatchNorm -> Activation) the normalisation is applied to the linear output of the Dense layer, so the values reaching the activation are well centred. For ReLU this means a healthy share of the neurons is active.

**Benefits for tabular data:** faster convergence, less sensitivity to the learning rate and a mild regularisation effect (the batch statistics add a little noise).

## 6.3 Train and compare: with vs without BatchNorm

In [ ]:
history_bn = model_bn.fit(X_train, y_train, epochs=50, batch_size=64,
                          validation_split=0.1, verbose=0)
print('BatchNorm model trained.')

In [ ]:
epochs = range(1, 51)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(epochs, history_base.history['loss'], color='#2C2C2C', linestyle='-', label='Baseline - train loss')
axes[0].plot(epochs, history_base.history['val_loss'], color='#2C2C2C', linestyle='--', label='Baseline - val loss')
axes[0].plot(epochs, history_bn.history['loss'], color='#CC0000', linestyle='-', label='BatchNorm - train loss')
axes[0].plot(epochs, history_bn.history['val_loss'], color='#CC0000', linestyle='--', label='BatchNorm - val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs, history_base.history['accuracy'], color='#2C2C2C', linestyle='-', label='Baseline - train acc')
axes[1].plot(epochs, history_base.history['val_accuracy'], color='#2C2C2C', linestyle='--', label='Baseline - val acc')
axes[1].plot(epochs, history_bn.history['accuracy'], color='#CC0000', linestyle='-', label='BatchNorm - train acc')
axes[1].plot(epochs, history_bn.history['val_accuracy'], color='#CC0000', linestyle='--', label='BatchNorm - val acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

fig.suptitle('Batch Normalization vs Baseline — Training Dynamics (Adult Income)', fontsize=13)
plt.savefig('plots/batchnorm_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

gap_base = history_base.history['accuracy'][-1] - history_base.history['val_accuracy'][-1]
gap_bn = history_bn.history['accuracy'][-1] - history_bn.history['val_accuracy'][-1]
print('Final train-validation accuracy gap - Baseline :', round(gap_base, 4))
print('Final train-validation accuracy gap - BatchNorm:', round(gap_bn, 4))

In [ ]:
y_prob_bn = model_bn.predict(X_test).flatten()
y_pred_bn = (y_prob_bn > 0.5).astype(int)

acc_bn = accuracy_score(y_test, y_pred_bn)
f1_bn = f1_score(y_test, y_pred_bn)

print('Test set results')
print('Baseline   - Accuracy:', round(acc_base, 4), ' F1 (class 1):', round(f1_base, 4))
print('BatchNorm  - Accuracy:', round(acc_bn, 4), ' F1 (class 1):', round(f1_bn, 4))

Look at the plot for two things: how fast the curves improve at the beginning, and how big the gap between the training and validation lines is at the end. The printed train-validation gap for both models is shown in the cell above. A smaller gap means less overfitting. The test accuracy and F1 (class 1) of both models are printed under the plot. These are the actual values from this run and they show whether BatchNorm helped on this dataset. BatchNorm does not always give a big improvement on a small network, so it is important to look at the numbers.

## 6.4 BatchNorm position experiment

In [ ]:
# (a) Dense -> Activation -> BatchNorm
keras.utils.set_random_seed(42)
model_bn_after = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, kernel_initializer='he_normal'),
    keras.layers.Activation('relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(64, kernel_initializer='he_normal'),
    keras.layers.Activation('relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(1, activation='sigmoid')
])
model_bn_after.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# (b) Dense -> BatchNorm -> Activation (canonical)
keras.utils.set_random_seed(42)
model_bn_before = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dense(64, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
model_bn_before.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_bn_after = model_bn_after.fit(X_train, y_train, epochs=30, batch_size=64,
                                      validation_split=0.1, verbose=0)
history_bn_before = model_bn_before.fit(X_train, y_train, epochs=30, batch_size=64,
                                        validation_split=0.1, verbose=0)
print('Both position models trained.')

In [ ]:
epochs30 = range(1, 31)
plt.figure(figsize=(10, 5))
plt.plot(epochs30, history_bn_before.history['val_accuracy'], color='#CC0000', label='BN before activation (canonical)')
plt.plot(epochs30, history_bn_after.history['val_accuracy'], color='#2C2C2C', linestyle='--', label='BN after activation')
plt.title('BatchNorm Position - Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend()
plt.savefig('plots/batchnorm_position.png', dpi=150, bbox_inches='tight')
plt.show()

before_last = np.mean(history_bn_before.history['val_accuracy'][-5:])
after_last = np.mean(history_bn_after.history['val_accuracy'][-5:])
print('Mean validation accuracy of last 5 epochs - BN before activation:', round(before_last, 4))
print('Mean validation accuracy of last 5 epochs - BN after activation :', round(after_last, 4))
print('Difference:', round(abs(before_last - after_last), 4))
if before_last > after_last:
    print('BN before activation performed better in this run.')
else:
    print('BN after activation performed better in this run.')

The original BN paper places normalisation before the activation, but many practitioners find that placing it after the activation works about the same or sometimes better. The cell above prints which order had the higher validation accuracy (average of the last 5 epochs) and how big the difference is.

I only trained each model once with one random seed and did not run any statistical test, so I cannot claim the difference is statistically significant. If the difference is only a fraction of a percent, it is most likely just run-to-run noise and both positions can be treated as performing about the same on the Adult Income data.

## 6.5 Inspect learned gamma and beta

In [ ]:
bn_layer = model_bn.layers[1]
gamma, beta, running_mean, running_var = bn_layer.get_weights()
print('gamma shape:', gamma.shape)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(range(len(gamma)), gamma, color='#CC0000')
axes[0].set_title('Learned gamma values - first BatchNorm layer (128 neurons)')
axes[0].set_xlabel('Neuron index')
axes[0].set_ylabel('gamma')

axes[1].bar(range(len(beta)), beta, color='#2C2C2C')
axes[1].set_title('Learned beta values - first BatchNorm layer (128 neurons)')
axes[1].set_xlabel('Neuron index')
axes[1].set_ylabel('beta')

plt.tight_layout()
plt.savefig('plots/bn_gamma_beta.png', dpi=150, bbox_inches='tight')
plt.show()

print('Neurons with |gamma| < 0.1 (almost gated off):', np.sum(np.abs(gamma) < 0.1))
print('Largest |gamma|:', round(np.max(np.abs(gamma)), 4), 'at neuron', np.argmax(np.abs(gamma)))
print('Mean gamma:', round(gamma.mean(), 4), ' Mean beta:', round(beta.mean(), 4))

**Gamma** is the learned scale and **beta** is the learned shift applied after the normalisation. At the start of training gamma = 1 and beta = 0 (no change), and training moves them away from these values.

- A neuron whose |gamma| is close to zero has been almost switched off, because its normalised output is multiplied by a very small number. The network found that this neuron's signal is not useful for predicting income. The number of such neurons in this run is printed above.
- A neuron with a large |gamma| gives a strong signal to the next layer, so it is one of the model's most important internal signals.
- Beta shifts the neuron's values up or down before ReLU. A negative beta pushes more values below zero, so the ReLU turns that neuron off for more samples. A positive beta does the opposite.

# Task 7: Optimizers - Convergence & Final Comparison

**Objective:** Compare five optimizers on the strongest configuration so far (ReLU, He Normal, BatchNorm, BCE), test learning rate sensitivity and train the final combined model.

**Expected output:** An optimizer convergence plot, a learning rate sensitivity plot, the final model results, a results table and ROC curves.

## 7.1 Build 5 models with different optimizers

Only the optimizer changes. Note that options (d) and (e) both mean Adam with the default values, so they are expected to give almost the same result. This confirms that the Keras default settings match the explicit ones.

In [ ]:
keras.utils.set_random_seed(42)
model_sgd = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                      use_batch_norm=True, loss='binary_crossentropy', optimizer='sgd')
history_sgd = model_sgd.fit(X_train, y_train, epochs=50, batch_size=64,
                            validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_sgd_mom = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                          use_batch_norm=True, loss='binary_crossentropy',
                          optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9))
history_sgd_mom = model_sgd_mom.fit(X_train, y_train, epochs=50, batch_size=64,
                                    validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_rmsprop = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                          use_batch_norm=True, loss='binary_crossentropy', optimizer='rmsprop')
history_rmsprop = model_rmsprop.fit(X_train, y_train, epochs=50, batch_size=64,
                                    validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_adam = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                       use_batch_norm=True, loss='binary_crossentropy', optimizer='adam')
history_adam = model_adam.fit(X_train, y_train, epochs=50, batch_size=64,
                              validation_split=0.1, verbose=0)

keras.utils.set_random_seed(42)
model_adam_explicit = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                                use_batch_norm=True, loss='binary_crossentropy',
                                optimizer=keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999))
history_adam_explicit = model_adam_explicit.fit(X_train, y_train, epochs=50, batch_size=64,
                                                validation_split=0.1, verbose=0)
print('All 5 optimizer models trained.')

## 7.2 Optimizer convergence plot

In [ ]:
histories_opt = {'SGD': history_sgd, 'SGD + momentum': history_sgd_mom, 'RMSprop': history_rmsprop,
                 'Adam': history_adam, 'Adam (explicit)': history_adam_explicit}
models_opt = {'SGD': model_sgd, 'SGD + momentum': model_sgd_mom, 'RMSprop': model_rmsprop,
              'Adam': model_adam, 'Adam (explicit)': model_adam_explicit}
colors_opt = {'SGD': '#7f7f7f', 'SGD + momentum': '#ff7f0e', 'RMSprop': '#2ca02c',
              'Adam': '#CC0000', 'Adam (explicit)': '#2C2C2C'}
styles_opt = {'SGD': '-', 'SGD + momentum': '--', 'RMSprop': '-.', 'Adam': '-', 'Adam (explicit)': ':'}

epochs = range(1, 51)
epoch_to_83 = {}

plt.figure(figsize=(12, 6))
i = 0
for name in histories_opt:
    val_acc = np.array(histories_opt[name].history['val_accuracy'])
    label = name
    crossed = np.where(val_acc >= 0.83)[0]
    if len(crossed) > 0:
        first_epoch = crossed[0] + 1
        epoch_to_83[name] = first_epoch
        label = f'{name} (83% at epoch {first_epoch})'
        plt.annotate('', xy=(first_epoch, 0.83), xytext=(first_epoch, 0.75 + i * 0.008),
                     arrowprops=dict(arrowstyle='->', color=colors_opt[name]))
    else:
        epoch_to_83[name] = None
        label = f'{name} (never reached 83%)'
    plt.plot(epochs, val_acc, color=colors_opt[name], linestyle=styles_opt[name], label=label)
    i = i + 1

plt.axhline(0.80, color='gray', linestyle='--', alpha=0.7)
plt.axhline(0.86, color='gray', linestyle='--', alpha=0.7)
plt.text(50, 0.801, '80%', ha='right', va='bottom', color='gray')
plt.text(50, 0.861, '86%', ha='right', va='bottom', color='gray')
plt.ylim(0.74, 0.88)
plt.title('Optimizer Comparison — Validation Accuracy over 50 Epochs (Adult Income ANN)')
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend(loc='lower right')
plt.savefig('plots/optimiser_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print('First epoch with validation accuracy >= 83%:', epoch_to_83)

In [ ]:
# test set F1 (class 1) for each optimizer
test_f1_opt = {}
for name in models_opt:
    y_prob = models_opt[name].predict(X_test, verbose=0).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    test_f1_opt[name] = f1_score(y_test, y_pred)
    print(name, '- test accuracy:', round(accuracy_score(y_test, y_pred), 4),
          ' test F1 (class 1):', round(test_f1_opt[name], 4))

best_opt_f1 = max(test_f1_opt, key=test_f1_opt.get)

# fastest = smallest first epoch above 83% (optimizers that never reached it are ignored)
reached = {}
for name in epoch_to_83:
    if epoch_to_83[name] is not None:
        reached[name] = epoch_to_83[name]
print()
print('Highest final class-1 F1 (test set):', best_opt_f1, round(test_f1_opt[best_opt_f1], 4))
if len(reached) > 0:
    fastest_opt = min(reached, key=reached.get)
    print('Fastest to 83% validation accuracy:', fastest_opt, 'at epoch', reached[fastest_opt])
else:
    print('No optimizer reached 83% validation accuracy in this run.')

## 7.3 Why Adam dominates

1. **Vanilla SGD:** It updates every weight with the same fixed learning rate: w = w - lr * gradient. It is simple, but it converges slowly and is sensitive to the learning rate. Our OHE features are sparse (mostly zeros), so many weights only get a gradient now and then, and one global learning rate is not suitable for all of them.

2. **SGD + Momentum:** It adds a velocity term: v = beta * v + lr * gradient, and updates with v. It remembers the direction of the past gradients, so it smooths out zig-zag movements and moves faster through flat regions of the loss. It still uses the same learning rate for all the parameters.

3. **RMSprop:** It divides the learning rate by a running average of the squared gradients of each parameter, sqrt(E[g^2]). Parameters with large gradients take smaller steps and parameters with small gradients take bigger steps. This per-parameter adaptation helps with sparse features.

4. **Adam:** It combines both ideas. It keeps a **first moment** m (the moving average of the gradients, like momentum) and a **second moment** v (the moving average of the squared gradients, like RMSprop). The update is theta = theta - lr * m_hat / (sqrt(v_hat) + epsilon).
   - **beta_1 = 0.9** controls how quickly the first moment forgets old gradients.
   - **beta_2 = 0.999** controls how quickly the second moment forgets old squared gradients.
   - **Bias correction:** m and v start at 0, so in the first steps they are too small. Dividing by (1 - beta^t) gives the corrected m_hat and v_hat and fixes this cold start.

5. **Adam (explicit):** This is the same as Adam with lr = 0.001, beta_1 = 0.9, beta_2 = 0.999. It confirms that the Keras defaults are the ones from the original paper (Kingma and Ba, 2015). Its curve is expected to be almost the same as the default Adam, and any small difference comes from the random operations of training.

The optimizer with the highest final class-1 F1 and the optimizer that converged fastest in this run are printed in the cell above the explanation, based on the actual results.

## 7.4 Learning rate sensitivity - SGD vs Adam

In [ ]:
sgd_histories = {}
for lr in [0.0001, 0.001, 0.01, 0.1]:
    keras.utils.set_random_seed(42)
    model_temp = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                           use_batch_norm=True, loss='binary_crossentropy',
                           optimizer=keras.optimizers.SGD(learning_rate=lr))
    sgd_histories[lr] = model_temp.fit(X_train, y_train, epochs=30, batch_size=64,
                                       validation_split=0.1, verbose=0)

adam_histories = {}
for lr in [0.0001, 0.001, 0.01]:
    keras.utils.set_random_seed(42)
    model_temp = build_ann(input_dim=X_train.shape[1], activation='relu', initializer='he_normal',
                           use_batch_norm=True, loss='binary_crossentropy',
                           optimizer=keras.optimizers.Adam(learning_rate=lr))
    adam_histories[lr] = model_temp.fit(X_train, y_train, epochs=30, batch_size=64,
                                        validation_split=0.1, verbose=0)
print('All 7 learning rate models trained.')

In [ ]:
epochs30 = range(1, 31)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for lr in sgd_histories:
    axes[0].plot(epochs30, sgd_histories[lr].history['val_loss'], label=f'SGD lr = {lr}')
axes[0].set_title('SGD')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation loss')
axes[0].legend()

for lr in adam_histories:
    axes[1].plot(epochs30, adam_histories[lr].history['val_loss'], label=f'Adam lr = {lr}')
axes[1].set_title('Adam')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation loss')
axes[1].legend()

fig.suptitle('Learning Rate Sensitivity — SGD vs Adam', fontsize=13)
plt.savefig('plots/optimiser_lr_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print('Final validation loss (epoch 30) and best validation loss:')
for lr in sgd_histories:
    v = sgd_histories[lr].history['val_loss']
    print('SGD  lr', lr, '- final:', round(v[-1], 4), ' best:', round(min(v), 4))
for lr in adam_histories:
    v = adam_histories[lr].history['val_loss']
    print('Adam lr', lr, '- final:', round(v[-1], 4), ' best:', round(min(v), 4))

sgd_finals = [sgd_histories[lr].history['val_loss'][-1] for lr in sgd_histories]
adam_finals = [adam_histories[lr].history['val_loss'][-1] for lr in adam_histories]
print()
print('Spread of final val loss across learning rates - SGD :', round(max(sgd_finals) - min(sgd_finals), 4))
print('Spread of final val loss across learning rates - Adam:', round(max(adam_finals) - min(adam_finals), 4))

The printed values above show what really happened for each learning rate. To read the plot:
- For SGD, a very small learning rate (0.0001) moves the loss down only slowly, while the larger ones (0.01 and 0.1) reach a lower loss faster. If a curve is jumping up and down or growing, that learning rate is too large for that optimizer.
- For Adam, the printed spread of the final validation loss across the learning rates tells how sensitive it is. A smaller spread means it is less sensitive to the learning rate.

The general reason is that Adam adapts the step size of every parameter using the second moment, so a not-so-good global learning rate is partly compensated. SGD uses the same step size for all the weights, so it depends much more on choosing a good learning rate. In this run SGD is compared only over 4 values and one seed, so the exact behaviour can change for other seeds. Adam with a large learning rate like 0.01 can also become noisy, so it is not completely immune to the learning rate.

## 7.5 Final Combined ANN - best configuration

The configuration from the project is ReLU, He Normal, BatchNorm and Adam. For the loss, the project says to use Weighted BCE only if it improved F1. I compare the weighted BCE model with the normal BCE model that has the same setup (ReLU + He Normal, from Task 4) and decide from the test F1 values.

In [ ]:
y_prob = model_he_normal.predict(X_test, verbose=0).flatten()
y_pred = (y_prob > 0.5).astype(int)
f1_bce_he = f1_score(y_test, y_pred)

print('Test F1 (class 1) - BCE, ReLU + he_normal    :', round(f1_bce_he, 4))
print('Test F1 (class 1) - Weighted BCE, same setup :', round(f1_wbce, 4))

if f1_wbce > f1_bce_he:
    class_weight_final = class_weight_dict
    final_loss_name = 'Weighted BCE'
    print('Weighted BCE improved F1, so the final model uses class weights.')
else:
    class_weight_final = None
    final_loss_name = 'BCE'
    print('Weighted BCE did not improve F1, so the final model uses plain BCE.')

print()
print('Best activation in Task 3 (val F1):', best_activation)
print('Best optimizer in Task 7.1 (test F1):', best_opt_f1)

In [ ]:
keras.utils.set_random_seed(42)
model_final = build_ann(input_dim=X_train.shape[1], hidden_units=[128, 64], activation='relu',
                        initializer='he_normal', use_batch_norm=True, optimizer='adam',
                        loss='binary_crossentropy')
history_final = model_final.fit(X_train, y_train, epochs=80, batch_size=64,
                                validation_split=0.1, class_weight=class_weight_final, verbose=0)

y_prob_final = model_final.predict(X_test).flatten()
y_pred_final = (y_prob_final > 0.5).astype(int)

print(classification_report(y_test, y_pred_final))
print('Final model - Accuracy      :', round(accuracy_score(y_test, y_pred_final), 4))
print('Final model - Precision (1) :', round(precision_score(y_test, y_pred_final), 4))
print('Final model - Recall (1)    :', round(recall_score(y_test, y_pred_final), 4))
print('Final model - F1 (1)        :', round(f1_score(y_test, y_pred_final), 4))
print('Final model - ROC-AUC       :', round(roc_auc_score(y_test, y_prob_final), 4))

## 7.6 Full results comparison table

In [ ]:
best_init = max(val_f1_init, key=val_f1_init.get)
print('Best initializer in Task 4 (val F1):', best_init)

# name, model, activation, init, loss, batchnorm, optimizer
models_list = [
    ['Baseline (Task 2)', model_base, 'relu', 'glorot_uniform', 'BCE', 'No', 'adam'],
    ['Best Activation (Task 3): ' + best_activation, models_act[best_activation], best_activation.lower(), 'glorot_uniform', 'BCE', 'No', 'adam'],
    ['Best Initialiser (Task 4): ' + best_init, models_init[best_init], 'relu', best_init, 'BCE', 'No', 'adam'],
    ['MSE (Task 5)', model_mse, 'relu', 'he_normal', 'MSE', 'No', 'adam'],
    ['Weighted BCE (Task 5)', model_wbce, 'relu', 'he_normal', 'Weighted BCE', 'No', 'adam'],
    ['Focal Loss (Task 5)', model_focal, 'relu', 'he_normal', 'Focal', 'No', 'adam'],
    ['BatchNorm (Task 6)', model_bn, 'relu', 'he_normal', 'BCE', 'Yes', 'adam'],
    ['Final Combined (Task 7)', model_final, 'relu', 'he_normal', final_loss_name, 'Yes', 'adam'],
]

rows = []
for item in models_list:
    y_prob = item[1].predict(X_test, verbose=0).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    rows.append({
        'Model': item[0], 'Activation': item[2], 'Init': item[3], 'Loss': item[4],
        'BatchNorm': item[5], 'Optimizer': item[6],
        'Test Acc': round(accuracy_score(y_test, y_pred), 4),
        'Prec(1)': round(precision_score(y_test, y_pred), 4),
        'Recall(1)': round(recall_score(y_test, y_pred), 4),
        'F1(1)': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
    })

df_results = pd.DataFrame(rows)
df_results.style.highlight_max(subset=['F1(1)', 'ROC-AUC'], color='#ffcccc')

In [ ]:
# save the results table as an image
fig, ax = plt.subplots(figsize=(20, 4))
ax.axis('off')
table_plot = ax.table(cellText=df_results.astype(str).values, colLabels=df_results.columns,
                      loc='center', cellLoc='center')
table_plot.auto_set_font_size(False)
table_plot.set_fontsize(8)
table_plot.scale(1, 1.8)
table_plot.auto_set_column_width(list(range(len(df_results.columns))))

# highlight the best F1(1) and ROC-AUC cells
best_f1_row = df_results['F1(1)'].idxmax()
best_auc_row = df_results['ROC-AUC'].idxmax()
f1_col = list(df_results.columns).index('F1(1)')
auc_col = list(df_results.columns).index('ROC-AUC')
table_plot[(best_f1_row + 1, f1_col)].set_facecolor('#ffcccc')
table_plot[(best_auc_row + 1, auc_col)].set_facecolor('#ffcccc')

plt.title('Full Results Comparison (Test Set)')
plt.savefig('plots/results_table.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves: Baseline, Weighted BCE, Final Combined
fpr_base, tpr_base, _ = roc_curve(y_test, y_pred_prob)
fpr_wbce, tpr_wbce, _ = roc_curve(y_test, y_prob_wbce)
fpr_final, tpr_final, _ = roc_curve(y_test, y_prob_final)

auc_base = roc_auc_score(y_test, y_pred_prob)
auc_wbce = roc_auc_score(y_test, y_prob_wbce)
auc_final = roc_auc_score(y_test, y_prob_final)

plt.figure(figsize=(8, 7))
plt.plot(fpr_base, tpr_base, color='#2C2C2C', label=f'Baseline (AUC = {auc_base:.4f})')
plt.plot(fpr_wbce, tpr_wbce, color='#1f77b4', linestyle='--', label=f'Weighted BCE (AUC = {auc_wbce:.4f})')
plt.plot(fpr_final, tpr_final, color='#CC0000', linestyle='-.', label=f'Final Combined (AUC = {auc_final:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Random guess')
plt.title('ROC Curves - Adult Income ANN (Test Set)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.savefig('plots/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# change in class-1 F1 compared with the baseline, for each technique
f1_of = {}
for i in range(len(df_results)):
    f1_of[df_results.loc[i, 'Model']] = df_results.loc[i, 'F1(1)']

f1_baseline = f1_of['Baseline (Task 2)']

changes = {
    'Activation': f1_of['Best Activation (Task 3): ' + best_activation] - f1_baseline,
    'Initialiser': f1_of['Best Initialiser (Task 4): ' + best_init] - f1_baseline,
    'Loss (Weighted BCE)': f1_of['Weighted BCE (Task 5)'] - f1_baseline,
    'Loss (Focal Loss)': f1_of['Focal Loss (Task 5)'] - f1_baseline,
    'Batch Normalization': f1_of['BatchNorm (Task 6)'] - f1_baseline,
    'Optimizer (best vs worst in Task 7.1)': max(test_f1_opt.values()) - min(test_f1_opt.values()),
}

print('Change in class-1 F1 compared with the baseline:')
for key in changes:
    print(key, ':', round(changes[key], 4))

best_technique = max(changes, key=changes.get)
print()
print('Largest positive effect on class-1 F1 in this run:', best_technique, round(changes[best_technique], 4))

In [ ]:
technique_reasons = {
    'Activation': 'Changing the activation only changes how the hidden layers bend the data. On a shallow network this usually does not change which side of the 0.5 threshold most people fall on, so it is not expected to help the minority class much.',
    'Initialiser': 'The initializer only changes the starting point of the weights. After 50 epochs the network usually reaches a similar place, so a big change in F1 would be unusual.',
    'Loss (Weighted BCE)': 'Class weights make mistakes on the >50K class more expensive, so the model moves its decision boundary to catch more of them. This raises recall of class 1 and, if the gain in recall is bigger than the loss in precision, F1 also goes up. It attacks the real problem of this dataset, which is the 75/25 imbalance.',
    'Loss (Focal Loss)': 'Focal Loss reduces the weight of easy examples and keeps the loss of hard examples, which shifts the training towards the hard, often minority class, samples. Its alpha value also changes the balance between the classes, so it acts on the imbalance directly.',
    'Batch Normalization': 'Batch Normalization stabilises the inputs of each layer and helps the network fit the data better, which can improve the quality of the predictions, including for the minority class.',
    'Optimizer (best vs worst in Task 7.1)': 'The optimizer changes how well and how fast the network reaches a good solution. A weak optimizer such as plain SGD may not have converged in 50 epochs, so the gap to a good optimizer can be large. Note that this number is the spread between the best and worst optimizer, not a gain over the baseline.',
}
print('Explanation for', best_technique, ':')
print(technique_reasons[best_technique])

**Which single technique had the largest individual impact on minority-class F1, and why?**

The answer for this run is printed in the two cells above: they show the change in class-1 F1 for each technique compared with the baseline, the technique with the largest positive change, and a short explanation of why that kind of technique can affect the minority class. The changes come from a single run with a single seed, so very small differences (for example 0.005 in F1) should not be treated as a real effect.

In general, on an imbalanced dataset like Adult Income, techniques that act directly on the class imbalance (class weights or focal loss) have the biggest chance to change minority-class F1. Techniques like the activation function, initializer, BatchNorm and optimizer mostly change how well and how quickly the network learns the same decision, so their effect on F1 is smaller. The table and the printed values above show whether that was true in this run.

In [ ]:
print('Saved plots:')
for file_name in sorted(os.listdir('plots')):
    print(file_name)